In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 996.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17

In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [4]:
train_data = datasets.AG_NEWS(split='train')
test_data = datasets.AG_NEWS(split='test')

In [5]:
tokenizer = get_tokenizer('basic_english')

In [6]:
# берём только 5 строк чтобы посмотреть какой столбец первым приходит
for i in list(train_data)[:5]:
    print(i)

(3, "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")
(3, 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.')
(3, "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.")
(3, 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.')
(3, 'Oil prices soar to all-time record, posing new menace t

In [7]:
def text_to_tokenizer(data):
    for label, text in data:
        yield tokenizer(text)

In [8]:
vocab = build_vocab_from_iterator(text_to_tokenizer(train_data), specials=["<unk>"])

In [9]:
vocab.set_default_index(vocab["<unk>"])

In [10]:
def change_label(label): # индексы классов начинается с 1 до 4. для обучение нужно приводить с 0 до 3
    return label - 1

def change_text(x):
    return [vocab[i] for i in tokenizer(x)]

In [11]:
def collate_batch(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(change_label(label))
        texts.append(torch.tensor(change_text(text), dtype=torch.int64))
    labels = torch.tensor(labels, dtype=torch.int64)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    return texts, labels

In [12]:
train_dataloader = DataLoader(list(train_data), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_data), batch_size=32, collate_fn=collate_batch)

In [13]:
class CheckNews(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, output_dim=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.lin = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        return self.lin(hidden[-1])

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
model = CheckNews(len(vocab)).to(device)

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [23]:
for epoch in range(50):
    model.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 9.64
Эпоха: 2 - Потери: 10.15
Эпоха: 3 - Потери: 10.42
Эпоха: 4 - Потери: 10.18
Эпоха: 5 - Потери: 10.9
Эпоха: 6 - Потери: 10.41
Эпоха: 7 - Потери: 8.25
Эпоха: 8 - Потери: 10.05
Эпоха: 9 - Потери: 9.49
Эпоха: 10 - Потери: 10.32
Эпоха: 11 - Потери: 9.1
Эпоха: 12 - Потери: 11.32
Эпоха: 13 - Потери: 8.78
Эпоха: 14 - Потери: 8.42
Эпоха: 15 - Потери: 8.48
Эпоха: 16 - Потери: 7.76
Эпоха: 17 - Потери: 7.61
Эпоха: 18 - Потери: 8.3
Эпоха: 19 - Потери: 9.32
Эпоха: 20 - Потери: 6.04
Эпоха: 21 - Потери: 9.42
Эпоха: 22 - Потери: 6.94
Эпоха: 23 - Потери: 8.1
Эпоха: 24 - Потери: 7.6
Эпоха: 25 - Потери: 6.72
Эпоха: 26 - Потери: 7.09
Эпоха: 27 - Потери: 6.61
Эпоха: 28 - Потери: 8.99
Эпоха: 29 - Потери: 5.08
Эпоха: 30 - Потери: 5.56


In [24]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 90.95%


In [26]:
from google.colab import files

torch.save(vocab, 'vocab_AG_NewsClassificationDataset.pth')
torch.save(model.state_dict(), 'model_AG_NewsClassificationDataset.pth')

files.download('vocab_AG_NewsClassificationDataset.pth')
files.download('model_AG_NewsClassificationDataset.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>